# Exercise: Data Preparation & Loading

Welcome to your first programming exercise for nanoGPT! In this notebook, you'll build the foundational components for loading and preparing text data for a language model. A solid data pipeline is the first step towards training a powerful model.

**In this exercise you will:**
*   Implement a simple character-level tokenizer to convert text into numerical format.
*   Create a `SimpleTextDataset` class to hold the tokenized data.
*   Implement a `get_batch` method to efficiently generate input-target pairs for training.

Let's get started!

In [ ]:
import torch
import random
from typing import List, Tuple

# We'll set a manual seed for reproducibility of our random batches
torch.manual_seed(42)

# ---- Helper Code ----
# You can ignore this cell. It's just a simple text corpus we'll use for testing.
TINY_SHAKESPEARE_CORPUS = """
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.
"""

### Exercise 1: Implement a Character-Level Tokenizer

Your first task is to create a simple tokenizer. A tokenizer's job is to convert raw text (a string) into a sequence of numbers that our model can understand. We'll start with a character-level tokenizer, which is the simplest kind. It works by mapping every unique character in our text to a unique integer.

You will implement two key methods:
-   `encode(s)`: Takes a string `s` and returns a list of integers representing that string.
-   `decode(l)`: Takes a list of integers `l` and returns the corresponding string.

**Hint:** You'll find the `self.stoi` (string-to-integer) and `self.itos` (integer-to-string) mappings, which are created for you in the `__init__` method, very useful!

In [ ]:
class CharacterTokenizer:
    """A simple character-level tokenizer."""
    def __init__(self, text: str):
        """Initializes the tokenizer and builds the vocabulary."""
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        # Create the string-to-integer and integer-to-string mappings
        self.stoi = { ch:i for i,ch in enumerate(self.chars) }
        self.itos = { i:ch for i,ch in enumerate(self.chars) }

    def encode(self, s: str) -> List[int]:
        """Converts a string to a list of integer tokens."""
        ### START CODE HERE ### (≈ 1 line)
        
        ### END CODE HERE ###

    def decode(self, l: List[int]) -> str:
        """Converts a list of integer tokens back to a string."""
        ### START CODE HERE ### (≈ 1 line)
        
        ### END CODE HERE ###

In [ ]:
# --- Test Cell for Exercise 1 ---

# 1. Initialize the tokenizer with our sample text
tokenizer = CharacterTokenizer(TINY_SHAKESPEARE_CORPUS)
print(f"Vocabulary: {''.join(tokenizer.chars)}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

# 2. Test the encode method
test_string = "Hello."
encoded_output = tokenizer.encode(test_string)
expected_encoded = [13, 22, 29, 29, 32, 5]
print(f"\nEncoding '{test_string}': {encoded_output}")
assert encoded_output == expected_encoded, f"Encode failed. Got {encoded_output}, expected {expected_encoded}"

# 3. Test the decode method
decoded_output = tokenizer.decode(expected_encoded)
print(f"Decoding {expected_encoded}: '{decoded_output}'")
assert decoded_output == test_string, f"Decode failed. Got '{decoded_output}', expected '{test_string}'"

# 4. Test the round-trip
round_trip_string = "You are all resolved."
encoded = tokenizer.encode(round_trip_string)
decoded = tokenizer.decode(encoded)
assert decoded == round_trip_string, "Round-trip encoding and decoding failed."

print("\n🎉 All tests passed for the CharacterTokenizer! Great work.")

### Exercise 2: Implement `get_batch`

Excellent! Now that you have a working tokenizer, you'll create a `SimpleTextDataset` class. This class will hold our entire text corpus as one long sequence of tokens.

Your main task is to implement the `get_batch` method. This method is crucial for training. It randomly samples starting positions from the dataset and creates a "batch" of input-target pairs.

For a language model, the task is to predict the *next* character. So, for a given input sequence `x`, the target sequence `y` is simply `x` shifted one position to the right.

**Example:**
If `block_size` is 8 and our text is "hello world", a possible sample could be:
-   `x` (input): "hello wo"
-   `y` (target): "ello wor"

Your implementation should return two PyTorch tensors, `x` and `y`, each with the shape `(batch_size, block_size)`.

In [ ]:
class SimpleTextDataset:
    """A simple dataset class to hold tokenized text and provide batches."""
    def __init__(self, text: str, tokenizer: CharacterTokenizer, block_size: int):
        self.tokenizer = tokenizer
        self.block_size = block_size
        encoded_text = self.tokenizer.encode(text)
        self.data = torch.tensor(encoded_text, dtype=torch.long)
        print(f"Dataset created with {len(self.data)} tokens.")

    def get_batch(self, batch_size: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Generates a batch of input (x) and target (y) sequences.

        Args:
            batch_size (int): The number of independent sequences in a batch.

        Returns:
            A tuple of (x, y) tensors, each of shape (batch_size, block_size).
        """
        # 1. Generate 'batch_size' random starting indices.
        #    Make sure you don't pick an index too close to the end of the data,
        #    or you won't have enough tokens for a full block!
        ### START CODE HERE ### (≈ 1 line)
        ix = ...
        ### END CODE HERE ###

        # 2. Create the input tensor 'x' by stacking the sequences.
        #    Each sequence should start at one of the random indices 'ix' and have length 'block_size'.
        ### START CODE HERE ### (≈ 1 line)
        x = ...
        ### END CODE HERE ###
        
        # 3. Create the target tensor 'y' by stacking the sequences.
        #    Each sequence in 'y' is the corresponding sequence in 'x' shifted by one.
        ### START CODE HERE ### (≈ 1 line)
        y = ...
        ### END CODE HERE ###

        return x, y

In [ ]:
# --- Test Cell for Exercise 2 ---

# 1. Initialize the dataset
block_size = 8
batch_size = 4
dataset = SimpleTextDataset(TINY_SHAKESPEARE_CORPUS, tokenizer, block_size)

# 2. Get a batch of data
# We set the seed again just before calling to ensure the random indices are the same every time.
torch.manual_seed(42)
x, y = dataset.get_batch(batch_size)

# 3. Test the shapes
print(f"\nShape of x: {x.shape}")
print(f"Shape of y: {y.shape}")
expected_shape = (batch_size, block_size)
assert x.shape == expected_shape, f"Shape of x is incorrect. Got {x.shape}, expected {expected_shape}"
assert y.shape == expected_shape, f"Shape of y is incorrect. Got {y.shape}, expected {expected_shape}"

# 4. Test the content relationship
# The first (block_size - 1) tokens of y should be identical to the last (block_size - 1) tokens of x.
print("\nVerifying that y is the shifted version of x...")
is_shifted_correctly = torch.equal(x[:, 1:], y[:, :-1])
assert is_shifted_correctly, "Content check failed: y is not correctly shifted from x."

print("Content relationship is correct!")
print("\n🎉 All tests passed for the SimpleTextDataset! You're ready to feed data to a model.")

### (Optional) Challenge: Train/Validation Split

In a real project, you would split your data into a training set (for learning) and a validation set (for evaluation).

Can you modify the `SimpleTextDataset` to handle this?
1.  Change the `__init__` method to accept a `split_ratio` (e.g., 0.9 for 90% train, 10% val). Inside `__init__`, split `self.data` into `self.train_data` and `self.val_data`.
2.  Modify `get_batch` to accept a `split` argument (e.g., a string, `'train'` or `'val'`).
3.  Based on the `split` argument, sample from either `self.train_data` or `self.val_data`.

This is an optional but very practical extension!

In [ ]:
# --- Integration Test: See it all in action! ---
# This cell combines your tokenizer and dataset to show you what the model will see.
# No need to code anything here, just run it and observe the output.

print("--- Integration Test ---")
# 1. Re-initialize tokenizer and dataset
integration_tokenizer = CharacterTokenizer(TINY_SHAKESPEARE_CORPUS)
integration_dataset = SimpleTextDataset(TINY_SHAKESPEARE_CORPUS, integration_tokenizer, block_size=16)

# 2. Get a batch
torch.manual_seed(1337) # Use a different seed for a new batch
x_batch, y_batch = integration_dataset.get_batch(batch_size=2)

# 3. Decode and print the first example in the batch
print("\n--- Example 1 from the batch ---\n")

input_text = integration_tokenizer.decode(x_batch[0].tolist())
target_text = integration_tokenizer.decode(y_batch[0].tolist())

print(f"Input (x):  '{input_text}'")
print(f"Target (y): '{target_text}'")

print("\nNotice how the target is exactly the input, shifted one character to the right.")
print("The model's goal will be: given 'B', predict 'e'; given 'Be', predict 'f', and so on.")
print("\nCongratulations on completing the data preparation exercise!")